# Ember — Soup Kernel Challenge on Kaggle T4

Development notebook. **No scored results yet.** Select GPU T4 ×2 and Internet on. The unchanged harness uses only GPU 0. Use the platform-created team repository. For final submission, pin its full commit hash and save a public executed notebook version.

Private repositories: add a read-only `GITHUB_TOKEN` through Kaggle Secrets and grant this notebook access. Optional `HF_TOKEN` reduces download rate limits. Never paste credentials into code cells.

In [ ]:
REPO_URL = "https://github.com/alikhan-kai/decentrathon.git"
COMMIT = ""  # Set the full final commit hash for submission.
SETUP_READY = SOURCE_READY = VALIDATION_READY = BENCHMARK_READY = False


In [ ]:
import os, subprocess, sys
SETUP_READY = SOURCE_READY = VALIDATION_READY = BENCHMARK_READY = False
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
gpu_names = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True)
print(gpu_names)
assert "T4" in gpu_names, "Select GPU T4 x2 before running"
# Remove Kaggle's unused torchao 0.10, incompatible with PEFT 0.20.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "soup-cli[train]==0.77.0", "transformers==5.16.1", "peft==0.20.0",
    "trl==0.29.1", "bitsandbytes==0.50.2", "accelerate==1.14.0", "pytest"], check=True)

SETUP_READY = True


In [ ]:
SOURCE_READY = VALIDATION_READY = BENCHMARK_READY = False
assert SETUP_READY, "Hardware and dependencies must be ready"
import base64, pathlib, shutil
assert REPO_URL.startswith("https://github.com/") and "<" not in REPO_URL, "Enter the platform repository URL"
repo = pathlib.Path("/kaggle/working/ember-repo")
clone_env = dict(os.environ)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        token = secrets.get_secret("GITHUB_TOKEN")
        header = base64.b64encode(("x-access-token:" + token).encode()).decode()
        clone_env.update(GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
            GIT_CONFIG_VALUE_0="Authorization: Basic " + header)
        del token, header
    except Exception:
        pass
    try:
        os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    except Exception:
        pass
except ImportError:
    pass
if repo.exists():
    shutil.rmtree(repo)
result = subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)],
                        env=clone_env, capture_output=True, text=True)
clone_env.clear()
assert result.returncode == 0, "Clone failed; check repository URL and read-only secret access"
if COMMIT:
    subprocess.run(["git", "-C", str(repo), "checkout", "-q", COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", actual_commit)
if COMMIT:
    assert actual_commit == COMMIT, "Use the full 40-character commit hash"
else:
    print("Development run: pin this commit before final submission")
def run(*args, env=None):
    subprocess.run([sys.executable, *args], cwd=repo, env=env, check=True)

SOURCE_READY = True


Run numerical checks before consuming the full training budget. CUDA checks must execute here; the CPU-only development results do not establish GPU correctness. All following commands stop on failure.

In [ ]:
VALIDATION_READY = BENCHMARK_READY = False
assert SOURCE_READY, "Source setup must succeed first"
run("-m", "pytest", "solution", "-q")
test_env = dict(os.environ, SOUP_APPLY_SOLUTION="1")
run("-m", "pytest", "tests", "-q", env=test_env)
run("task/prepare_data.py")

VALIDATION_READY = True


These are the official cold-cache repeated measurements, in base/solution/solution/base order. Do not run component benchmarks or other GPU jobs concurrently. Compilation and initialization remain included by the unchanged harness.

In [ ]:
BENCHMARK_READY = False
assert VALIDATION_READY, "CUDA and regression tests must pass first"
run("harness/run_task.py", "--repeats", "2")
run("harness/check_rules.py")
run("harness/score.py", "runs/result.json")

BENCHMARK_READY = True


In [ ]:
import json
from IPython.display import FileLink, display
assert SOURCE_READY and (repo / "runs").is_dir(), "No run logs exist to export"
print("Diagnostic artifacts; exporting them does not imply all checks passed.")
checks_path = repo / "runs/checks.json"
if checks_path.exists():
    print(json.dumps(json.loads(checks_path.read_text()), indent=2))
archive = shutil.make_archive("/kaggle/working/ember-run-artifacts", "zip", root_dir=repo, base_dir="runs")
display(FileLink(archive))


Submission checklist: final platform commit, all official checks passing, measured results copied accurately into README and SUBMISSION, public saved notebook version linked to that commit, presentation and demo video (≤3 minutes). A GPU component timing alone is not a scored result.